# Control con Función de Política: Actor-Critic (TD)

- Estudiante 1: `nombre apellido - nro estudiante`
- Estudiante 2: `nombre apellido - nro estudiante`
- Estudiante 3: `nombre apellido - nro estudiante`

En este notebook implementaremos el algoritmo **Actor-Critic con TD (episódico)**, una técnica que combina la estimación de valores y la mejora de política en paralelo. A diferencia de los métodos puramente basados en valores como Q-learning o SARSA, los métodos Actor-Critic mantienen dos funciones diferenciadas: el **actor**, que representa la política, y el **crítico**, que estima el valor de los estados (o pares estado-acción).

Este enfoque aprovecha la estabilidad de las actualizaciones del valor para guiar el aprendizaje de la política directamente, haciendo uso del gradiente de política y del error de TD. Esto permite manejar entornos con espacios de acción continuos o políticas estocásticas.

## Objetivos

* Implementar el algoritmo **Actor-Critic (episódico)** utilizando PyTorch.

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

In [ ]:
import os

if os.name == 'posix' and os.uname().sysname == 'Darwin':
    # Set the path to ffmpeg for macOS, replace with your actual path
    os.environ["IMAGEIO_FFMPEG_EXE"] = "/opt/homebrew/bin/ffmpeg"

## Definir constantes y funciones auxiliares

In [ ]:
DEVICE = 'cpu'  # por defecto, usamos la CPU
if torch.cuda.is_available():  # si hay una GPU disponible (y cuda está instalado)
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():  # si hay un MPS disponible (Metal Performance Shaders)
    DEVICE = 'mps'

Podemos cambiar el ambiente para probar otros casos.

In [ ]:
ENVS = ["MountainCar-v0", "CartPole-v1"]
ENV_NAME = ENVS[1]  # Cambia esto para probar con otros entornos

In [ ]:
def get_env(env_name, record_video=False, record_every=1, folder="./videos" ):
    """
    Create the MountainCar environment with optional video recording and statistics.
    Args:
        env_name (str): Name of the environment to create.
        record_video (bool): Whether to record video of the episodes.
        record_every (int): Frequency of recording episodes.
        folder (str): Folder to save the recorded videos.
    Returns:
        env (gym.Env): The MountainCar environment.
        
    See also:
        https://gymnasium.farama.org/introduction/record_agent/
    """
    # Initialise the environment
    env = gym.make(env_name, render_mode="rgb_array")

    if record_video:
        env = RecordVideo(env, video_folder=folder,
                    episode_trigger=lambda x: x % record_every == 0)
    
    return env

Exploramos minimamente el entorno (Observation space y Action space) para entender cómo interactuar con él.

In [ ]:
env = get_env(ENV_NAME)
print(f"Environment: {ENV_NAME}")
print(f"Observation space: {env.observation_space}")
print(f"Observation space shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")

INPUT_DIM = env.observation_space.shape[0]
N_ACTIONS = env.action_space.n

Definimos una función que nos pase nuestra observación a un tensor de PyTorch, lo cual es necesario para trabajar con redes neuronales.

In [ ]:
def process_state(obs, device=DEVICE):
    return torch.tensor(obs, device=device).unsqueeze(0)

## Redes (función de política y función de valor)

Vamos a definir dos redes neuronales: una para la política (actor) y otra para el valor (crítico). Ambas redes tendrán una arquitectura simple. Lo importante es que la red de política saldrá una distribución de probabilidad sobre las acciones, mientras que la red de valor saldrá un valor escalar para el estado actual.

> La red de política (actor) se encargará de seleccionar acciones basadas en la política aprendida, mientras que la red de valor (crítico) evaluará el estado actual y proporcionará retroalimentación al actor. Es importate e ultizar una función de activación adecuada para la salida de la red de política, como `softmax`.

In [ ]:
tensor_actor_test = torch.tensor([1.0, 2.0, 3.0], device=DEVICE).unsqueeze(0)  # Añadimos una dimensión para simular un batch
print(f"{torch.nn.Softmax(dim=-1)(tensor_actor_test)}") 
print(f"{torch.nn.Softmax(dim=-1)(tensor_actor_test).sum()}")

### Definir la red de política (actor)

In [ ]:
class ActorCNN(nn.Module):
    def __init__(self, input_dim, n_actions):
        super().__init__()
        pass
    
    def forward(self, x):
        pass

actor_model = ActorCNN(INPUT_DIM, N_ACTIONS).to(DEVICE)

summary(actor_model, input_size=(1, INPUT_DIM), device=DEVICE)

Probamos con un tensor

In [ ]:
obs, _ = env.reset() # obtenermos un estado inicial del entorno
tensor_obs_test = process_state(obs, device=DEVICE) # lo pasamos a un tensor
actor_model(tensor_obs_test) # lo pasamos por la red neuronal (debería devolver una distribución de probabilidad sobre las acciones)

### Definir la red valor (crítico)

In [ ]:
class CriticCNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        pass
    
    def forward(self, x):
        pass

critic_model = CriticCNN(INPUT_DIM).to(DEVICE)

summary(critic_model, input_size=(1, INPUT_DIM), device=DEVICE)

In [ ]:
obs, _ = env.reset() # obtenermos un estado inicial del entorno
tensor_obs_test = process_state(obs, device=DEVICE) # lo pasamos a un tensor
critic_model(tensor_obs_test) # lo pasamos por la red neuronal (esto debería devolver un valor de estado, no una distribución de probabilidad)

## Sampleo de acciones
Para samplear acciones vamos a utilizar la red de política. Dado que la salida de la red de política es una distribución de probabilidad, utilizaremos `torch.distributions.Categorical` para muestrear acciones basadas en esta distribución.

El paquete `torch.distributions` nos permite trabajar con distribuciones probabilísticas de manera sencilla, pudienendo muestrear acciones y calcular log-probabilidades de las acciones seleccionadas.

Dichos tensores (log-probabilidades) contienen los gradientes necesarios para actualizar la red de política durante el entrenamiento.

Ver [torch.distributions](https://docs.pytorch.org/docs/stable/distributions.html) y [Categorical](https://docs.pytorch.org/docs/stable/distributions.html#torch.distributions.categorical.Categorical).

In [ ]:
prob_tensor = torch.tensor([0.1, 0.2, 0.7], device=DEVICE, requires_grad=True).unsqueeze(0)  # Simulamos una distribución de probabilidad
distribtion = torch.distributions.Categorical(prob_tensor)
print(f"Probabilidades: {prob_tensor}")
action = distribtion.sample()  # Muestreamos una acción de la distribución
print(f"Acción muestreada: {action}")
print(f"Log-probabilidades: {distribtion.log_prob(action)}")

## Algoritmo Actor-Critic

**Input:**  
- una parametrización diferenciable de la política $\pi(a|s, \theta)$  
- una parametrización diferenciable de la función de valor de estado $\hat{v}(s, \mathbf{w})$  

**Parámetros:** tasas de aprendizaje $\alpha^\theta > 0$, $\alpha^{\mathbf{w}} > 0$

**Inicializar:**  
- parámetros de la política $\theta \in \mathbb{R}^{d'}$
- pesos del valor de estado $\mathbf{w} \in \mathbb{R}^d$ (por ejemplo, a $0$)

$$
\begin{array}{l}
\textbf{Loop forever (para cada episodio):} \\
\quad \text{Inicializar } S \text{ (primer estado del episodio)} \\
\quad I \leftarrow 1 \\
\quad \textbf{Loop mientras } S \text{ no sea terminal (para cada paso de tiempo):} \\
\quad\quad A \sim \pi(\cdot|S, \theta) \\
\quad\quad \text{Tomar acción } A, \text{ observar } S', R \\
\quad\quad \delta \leftarrow R + \gamma \hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w}) \quad (\text{si } S' \text{ es terminal, } \hat{v}(S', \mathbf{w}) \doteq 0) \\
\quad\quad \mathbf{w} \leftarrow \mathbf{w} + \alpha^{\mathbf{w}} \delta \nabla \hat{v}(S, \mathbf{w}) \\
\quad\quad \theta \leftarrow \theta + \alpha^{\theta} I \delta \nabla \ln \pi(A|S, \theta) \\
\quad\quad I \leftarrow \gamma I \\
\quad\quad S \leftarrow S' \\
\end{array}
$$

In [ ]:
def select_action(action_probs_tensor):
    """
    Muestrea una acción a partir de una distribución categórica sobre las acciones.

    Parámetros:
        action_probs_tensor: tensor con las probabilidades de cada acción
                             (salida del actor, ya pasada por softmax).

    Retorna:
        action (int): la acción muestreada, como entero (para pasársela al env).
        log_prob_action (tensor): log π(a|s) de la acción elegida, conservando
                                  el grafo de cómputo para poder backprop-earlo en el actor.
    """
    raise NotImplementedError("select_action")


def train(env, actor_net, critic_net, process_state_fn, num_episodes=10_000, actor_lr=.0001, critic_lr=.0001, gamma=.99):
    """
    Entrena con One-Step Actor-Critic (versión episódica, Sutton & Barto §13.5).
    El actor aprende la política π(a|s) y el critic aprende la función de valor V(s).
    Las actualizaciones son online: una por cada paso del entorno (no esperamos al fin del episodio).

    Parámetros:
        env:              entorno tipo Gymnasium (debe exponer reset() y step()).
        actor_net:        red de política. Recibe el estado y devuelve probabilidades de acción.
        critic_net:       red de valor. Recibe el estado y devuelve un escalar V(s).
        process_state_fn: función que transforma la observación cruda del env en
                          el tensor que esperan las redes (normalización, reshape, etc.).
        num_episodes:     cantidad de episodios de entrenamiento.
        actor_lr:         learning rate del optimizador del actor.
        critic_lr:        learning rate del optimizador del critic.
        gamma:            factor de descuento.

    Retorna:
        episode_rewards (list[float]): retorno total (sin descontar) de cada episodio,
                                       útil para graficar la curva de aprendizaje.
    """
    raise NotImplementedError("train")

def play_episodes(env, actor_net, process_state_fn, num_episodes=5):
    """
    Corre episodios usando la política ya entrenada (sin aprendizaje).
    Pensado para visualizar/evaluar el comportamiento del agente.

    Parámetros:
        env:              entorno (idealmente con render_mode='human' si querés verlo).
        actor_net:        red de política entrenada.
        process_state_fn: misma función de preprocesamiento usada en el entrenamiento.
        num_episodes:     cantidad de episodios a ejecutar.
    """
    raise NotImplementedError("play_episodes")

In [ ]:
actor_net = ActorCNN(INPUT_DIM, N_ACTIONS).to(DEVICE)
critic_net = CriticCNN(INPUT_DIM).to(DEVICE)

In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=100, folder="./videos/train" )
train(env, actor_net, critic_net, process_state_fn=process_state, num_episodes=2_000, actor_lr=0.0001, critic_lr=0.0005)

In [ ]:
env = get_env(ENV_NAME, record_video=True, record_every=1, folder="./videos/test" )
play_episodes(env, actor_net, process_state_fn=process_state, num_episodes=5)